In [ ]:
!pip install -U langchain-google-genai langsmith

In [ ]:
import os
from google.colab import userdata

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API")
os.environ["LANGSMITH_PROJECT"] = "AI_TRADING_AGENT"

In [ ]:
!pip install langchain langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('API_KEY')

model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a stock trading analyst."
)

In [ ]:
import requests
from google.colab import userdata

twelvedata_api_key = userdata.get('TWELVE_DATA')

url = "https://api.twelvedata.com/price"
params = {
    "symbol": "AAPL",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'price': '358.14'}


In [ ]:
from langchain.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current market price for a stock. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/price"
    params = {"symbol": ticker, "apikey": twelvedata_api_key}
    response = requests.get(url, params=params)
    data = response.json()
    if "price" not in data:
        return f"Could not fetch price for {ticker}: {data.get('message', 'Unknown error')}"
    return f"{ticker} current price: ${float(data['price']):.2f}"

In [ ]:
print(get_stock_price.invoke({"ticker": "AAPL"}))

AAPL current price: $358.14


Building the Price History Tool


In [ ]:
url = "https://api.twelvedata.com/time_series"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "outputsize": "10",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock'}, 'values': [{'datetime': '2026-11-13', 'open': '357.93', 'high': '360.815', 'low': '357.32', 'close': '358.05', 'volume': '620767'}, {'datetime': '2026-11-12', 'open': '357.81000', 'high': '359.37000', 'low': '354.41000', 'close': '354.88000', 'volume': '31796600'}, {'datetime': '2026-11-11', 'open': '349.54999', 'high': '355.91000', 'low': '349.32999', 'close': '355.63000', 'volume': '34453500'}, {'datetime': '2026-11-10', 'open': '351.35001', 'high': '353.87000', 'low': '346.45999', 'close': '348.51001', 'volume': '29756100'}, {'datetime': '2026-11-09', 'open': '345', 'high': '352.32999', 'low': '344.78000', 'close': '350.69000', 'volume': '48753100'}, {'datetime': '2026-11-06', 'open': '338.73001', 'high': '345.79001', 'low': '338.29999', 'close': '342.37000', 'volume': '44703800'}, {'datetime': '2026-11-05', 'o

In [ ]:
@tool
def get_price_history(ticker: str) -> str:
    """Get the last 10 days of closing prices for a stock. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/time_series"
    params = {
        "symbol": ticker,
        "interval": "1day",
        "outputsize": "10",
        "apikey": twelvedata_api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    if "values" not in data:
        return f"Could not fetch price history for {ticker}: {data.get('message', 'Unknown error')}"
    prices = [
        f"  {item['datetime']}: ${float(item['close']):.2f}"
        for item in reversed(data["values"])
    ]
    return f"{ticker} closing prices (last 10 days):\n" + "\n".join(prices)

In [ ]:
print(get_price_history.invoke({"ticker": "AAPL"}))

AAPL closing prices (last 10 days):
  2026-11-02: $338.32
  2026-11-03: $331.90
  2026-11-04: $349.51
  2026-11-05: $344.17
  2026-11-06: $342.37
  2026-11-09: $350.69
  2026-11-10: $348.51
  2026-11-11: $355.63
  2026-11-12: $354.88
  2026-11-13: $358.05


RSI
— Relative Strength Index NEARLY 70 OR ABOVE 70 MANY PEOPLE BOUGHT
30-70 NORMAL RANGE
<30 OVERSOLD

In [ ]:
url = "https://api.twelvedata.com/rsi"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "time_period": "14",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'RSI - Relative Strength Index', 'series_type': 'close', 'time_period': 14}}, 'values': [{'datetime': '2026-11-13', 'rsi': '61.79684'}, {'datetime': '2026-11-12', 'rsi': '59.68862'}, {'datetime': '2026-11-11', 'rsi': '60.42115'}, {'datetime': '2026-11-10', 'rsi': '55.61989'}, {'datetime': '2026-11-09', 'rsi': '57.60670'}, {'datetime': '2026-11-06', 'rsi': '51.46215'}, {'datetime': '2026-11-05', 'rsi': '53.0055666'}, {'datetime': '2026-11-04', 'rsi': '57.77916'}, {'datetime': '2026-11-03', 'rsi': '41.70189'}, {'datetime': '2026-11-02', 'rsi': '47.87307'}, {'datetime': '2026-10-30', 'rsi': '55.34831'}, {'datetime': '2026-10-29', 'rsi': '61.38541'}, {'datetime': '2026-10-28', 'rsi': '56.10903'}, {'datetime': '2026-10-27', 'rsi': '61.099271'}, {'datetime': '2026-10-26', 'rsi': '57.69063'}, {'datet

In [ ]:
@tool
def calculate_rsi(ticker: str) -> str:
    """Calculate the current RSI for a stock to check if it is overbought or oversold. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/rsi"
    params = {
        "symbol": ticker,
        "interval": "1day",
        "time_period": "14",
        "apikey": twelvedata_api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    if "values" not in data:
        return f"Could not fetch RSI for {ticker}: {data.get('message', 'Unknown error')}"
    rsi = float(data["values"][0]["rsi"])
    if rsi > 70:
        signal = "Overbought — price may drop soon"
    elif rsi < 30:
        signal = "Oversold — price may rise soon"
    else:
        signal = "Normal"
    return f"{ticker} RSI(14): {rsi:.1f} → {signal}"

In [ ]:
print(calculate_rsi.invoke({"ticker": "AAPL"}))

AAPL RSI(14): 61.8 → Normal


In [ ]:
url = "https://api.twelvedata.com/sma"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "time_period": "50",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'SMA - Simple Moving Average', 'series_type': 'close', 'time_period': 50}}, 'values': [{'datetime': '2026-11-13', 'sma': '339.072002'}, {'datetime': '2026-11-12', 'sma': '338.47520'}, {'datetime': '2026-11-11', 'sma': '337.87680'}, {'datetime': '2026-11-10', 'sma': '337.26680'}, {'datetime': '2026-11-09', 'sma': '336.63360'}, {'datetime': '2026-11-06', 'sma': '336.013801'}, {'datetime': '2026-11-05', 'sma': '335.45800'}, {'datetime': '2026-11-04', 'sma': '334.84360'}, {'datetime': '2026-11-03', 'sma': '334.051401'}, {'datetime': '2026-11-02', 'sma': '333.62020'}, {'datetime': '2026-10-30', 'sma': '333.040801'}, {'datetime': '2026-10-29', 'sma': '332.37420'}, {'datetime': '2026-10-28', 'sma': '331.73260'}, {'datetime': '2026-10-27', 'sma': '331.056600'}, {'datetime': '2026-10-26', 'sma': '330.2

In [ ]:
@tool
def get_sma(ticker: str) -> str:
    """Get the 50-day and 200-day Simple Moving Averages to check the long-term price trend. Example: ticker='AAPL'"""
    base_params = {
        "symbol": ticker,
        "interval": "1day",
        "apikey": twelvedata_api_key
    }

    response_50 = requests.get(
        "https://api.twelvedata.com/sma",
        params={**base_params, "time_period": "50"}
    )
    response_200 = requests.get(
        "https://api.twelvedata.com/sma",
        params={**base_params, "time_period": "200"}
    )

    data_50 = response_50.json()
    data_200 = response_200.json()

    if "values" not in data_50 or "values" not in data_200:
        return f"Could not fetch SMA for {ticker}"

    sma_50 = float(data_50["values"][0]["sma"])
    sma_200 = float(data_200["values"][0]["sma"])

    if sma_50 > sma_200:
        trend = "Golden Cross — long-term uptrend (Bullish)"
    else:
        trend = "Death Cross — long-term downtrend (Bearish)"

    return f"{ticker} 50-day SMA: ${sma_50:.2f} | 200-day SMA: ${sma_200:.2f} | {trend}"

In [ ]:
system_prompt = """You are a stock trading analyst.

When asked to analyse a stock, follow these steps in order:
1. Get the current price → get_stock_price
2. Get the recent price history → get_price_history
3. Calculate RSI → calculate_rsi
4. Get the 50-day and 200-day SMA → get_sma

Then output your analysis in this format:

DECISION: BUY / SELL / HOLD
CONFIDENCE: High / Medium / Low

Current Price: $[price]
Recent trend: [describe the direction from price history]

RSI(14): [value] → [Overbought / Neutral / Oversold]
SMA: [Golden Cross / Death Cross] — [short explanation]

Rationale: [2 sentences explaining the decision based on the signals above]

Do not skip any tool. Always run all four before giving a decision."""

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_stock_price, get_price_history, calculate_rsi, get_sma],
    system_prompt=system_prompt
)

In [ ]:

response = agent.invoke({
    "messages": [{"role": "user", "content": "Should I buy Apple stock?"}]
})
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I am unable to fetch the recent price history for AAPL due to an API credit limit. Therefore, I cannot complete the analysis and provide a recommendation at this time. Please try again in a minute.', 'extras': {'signature': 'CrgEAWkUfRMCcTVq0hSHrFQIoyXbisYeSGg7I9J7Aulkr3sGQgFEbKA0PvHmFVBrM0PDNhDw/HILTyUAxDnEMEvLTfloWPtD5tzHF1ABkuKsE8ikTkDG+kdbtiT0AK7XMNHGCXBG3kQfZHQd8a46J+cmFJEvKWcqP94BzsFYsxcPQgZYc69JBemq2S5GKvCCLbnCHscpxrT33IAh0sBTqXAVW6HH1eTAl3q7l5KdYy+MaDUp4cBK8HcY4qaBJRFjGmK+eEd+SYAqlwpylLwxDjo++HiCYRyz31vHXNuyuibZJf4RsdDBHSsVoalOjR+T5sETVUo4108pnzQPuputUNZKrHcGssGBon9lIIdrs619zcgzxglWJR8sRy83iAYjDZqgWTb++Fm3iUM8lLBXezfouFuKx7Ha/T+wN237mZqm/MsAm8dzQvQTRubduyYj7KNsl1T74FTmheDL0cqedlD7EMjCfw986QpVXQnmojeCK+udrUznbMe3QaBvIxFIfrOYJT9JieyZFtWDIPky5ZProxDalK4YFjSLhbibYHUMDk+RwMLB2By8JMbUcvoqikGXK6zGTOl84owitgLHJNrXKPL/wU7wPhnbTRQoKcGSMYRtnvEi9BUc/EiwitVrZgI6dyoo/+I+l7bxfNfIcfvOdbybhbr859/DPME1sjfhbVlU5YOylg5s9vyWAsL71Q8eHoOeMYIg2zCuYwo7xMXLt0IGQ2qAi1NongM/YPNL

this is ok but there is some thing that no chat can predict     that is NEWS



1.  Price trending up + RSI normal + News is positive	BUY
2.  Price trending up + RSI overbought (too many buyers)	HOLD (wait for price to settle)



1.   News is negative (bad headlines)	HOLD or SELL
2.   Price trending down + Death Cross	SELL (if we own the stock)








we are adding a news releated tool to our agent and take the descision

NEWS DATA API Finnhub

In [ ]:
!pip install finnhub-python

In [ ]:
import finnhub
from google.colab import userdata

finnhub_client = finnhub.Client(api_key=userdata.get('FINN_HUB_API'))

In [ ]:
from datetime import datetime, timedelta

today = datetime.now().strftime("%Y-%m-%d")
week_ago = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

news = finnhub_client.company_news("AAPL", _from=week_ago, to=today)
print(news[:2])

[{'category': 'company', 'datetime': 1794584582, 'headline': 'Forget the "Magnificent Seven." Berkshire Hathaway Could Be the Better Long-Term Bet.', 'id': 143803787, 'image': 'https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png', 'related': 'AAPL', 'source': 'Yahoo', 'summary': 'The stock has averaged a 20% annual return over the past 60 years.', 'url': 'https://finnhub.io/api/news?id=62f45ca7e59eeb458e57ddd7f0ca7d1c6005c02bf1b5436557d524510b496987'}, {'category': 'company', 'datetime': 1794584521, 'headline': 'Apple Is the World’s Largest Company, and Nvidia Is Second. Here’s What $10,000 Invested in Each Today Could Be Worth by the End of Next Year.', 'id': 143803789, 'image': 'https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png', 'related': 'AAPL', 'source': 'Yahoo', 'summary': 'The iPhone maker now has a commanding market cap lead over the data center chipmaker.', 'url': 'https://finnhub.io/api/news?id=55d6dfc709424f81412bee1e3c70061407ad551045247a5

In [ ]:
from langchain.tools import tool
from datetime import datetime, timedelta

@tool
def get_company_news(ticker: str) -> str:
    """Get the latest 5 news headlines for a stock from the past 7 days. Example: ticker='AAPL'"""
    today = datetime.now().strftime("%Y-%m-%d")
    week_ago = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    news = finnhub_client.company_news(ticker, _from=week_ago, to=today)
    if not news:
        return f"No recent news found for {ticker} in the past 7 days."
    headlines = [f"  • {item['headline']}" for item in news[:5]]
    return f"Latest news for {ticker}:\n" + "\n".join(headlines)

In [ ]:
print(get_company_news.invoke('TSLA'))

Latest news for TSLA:
  • Here’s What Needs to Happen for Tesla’s Bull Case to Materialize and Push Past $500 Next Year
  • Forget the "Magnificent Seven." Berkshire Hathaway Could Be the Better Long-Term Bet.
  • SpaceX Flight 15: Morgan Stanley Calls Upcoming Starship Test One Of The Most ‘Significant Catalysts’ In Company’s 25-Year History
  • Tesla-SpaceX merger announcement as early as December: Bloomberg
  • SpaceX and Tesla merger talks advance as December announcement considered


ALPACA FOR PAPER TRADING

In [ ]:
import requests
from google.colab import userdata

ALPACA_BASE_URL = "https://paper-api.alpaca.markets"

alpaca_headers = {
    "APCA-API-KEY-ID": userdata.get("ALPACA_KEY"),
    "APCA-API-SECRET-KEY": userdata.get("ALPACA_SECRET")
}

response = requests.get(f"{ALPACA_BASE_URL}/v2/account", headers=alpaca_headers)
print(response.json())

{'id': '0d127940-6b50-4aa4-8980-65218ed175ad', 'admin_configurations': {}, 'user_configurations': None, 'account_number': 'PA3TA69A3', 'status': 'ACTIVE', 'crypto_status': 'ACTIVE', 'options_approved_level': 3, 'options_trading_level': 3, 'currency': 'USD', 'buying_power': '399134.09', 'regt_buying_power': '199280.46', 'effective_buying_power': '399134.09', 'non_marginable_buying_power': '99640.23', 'options_buying_power': '99640.23', 'cash': '99282', 'accrued_fees': '0', 'portfolio_value': '99998.46', 'trading_blocked': False, 'transfers_blocked': False, 'account_blocked': False, 'created_at': '2026-11-13T10:30:46.894397Z', 'trade_suspended_by_user': False, 'multiplier': '4', 'shorting_enabled': True, 'equity': '99998.46', 'last_equity': '100000', 'long_market_value': '716.46', 'short_market_value': '0', 'position_market_value': '716.46', 'initial_margin': '358.23', 'maintenance_margin': '214.94', 'last_maintenance_margin': '0', 'sma': '0', 'balance_asof': '2026-11-12', 'crypto_tier':

NOW NEED TO ADD BUY AND SELL TOOLS TO AN AGENT

PLACE ORDER

In [ ]:
response = requests.post(
    f"{ALPACA_BASE_URL}/v2/orders",
    json={
        "symbol": "AAPL",
        "qty": "1",
        "side": "buy",
        "type": "market",
        "time_in_force": "day"
    },
    headers={**alpaca_headers, "Content-Type": "application/json"}
)
print(response.json())

{'id': '4ad09161-8238-4329-b02f-d8c5293ba9a7', 'client_order_id': '409badd3-2ea9-4c70-a3e9-b734177aae98', 'created_at': '2026-11-13T17:36:47.165952598Z', 'updated_at': '2026-11-13T17:36:47.167222643Z', 'submitted_at': '2026-11-13T17:36:47.165952598Z', 'filled_at': None, 'expired_at': None, 'canceled_at': None, 'failed_at': None, 'replaced_at': None, 'replaced_by': None, 'replaces': None, 'asset_id': 'b0b6dd9d-8b9b-48a9-ba46-b9d54906e415', 'symbol': 'AAPL', 'asset_class': 'us_equity', 'notional': None, 'qty': '1', 'filled_qty': '0', 'filled_avg_price': None, 'order_class': '', 'order_type': 'market', 'type': 'market', 'side': 'buy', 'position_intent': 'buy_to_open', 'time_in_force': 'day', 'limit_price': None, 'stop_price': None, 'status': 'pending_new', 'extended_hours': False, 'legs': None, 'trail_percent': None, 'trail_price': None, 'hwm': None, 'subtag': None, 'source': None, 'expires_at': '2026-11-13T21:00:00Z'}


In [ ]:
@tool
def place_order(ticker: str, side: str, quantity: int) -> str:
    """Place a paper trade market order on Alpaca. side must be 'buy' or 'sell'. Example: ticker='AAPL', side='buy', quantity=1"""
    response = requests.post(
        f"{ALPACA_BASE_URL}/v2/orders",
        json={
            "symbol": ticker,
            "qty": str(quantity),
            "side": side.lower(),
            "type": "market",
            "time_in_force": "day"
        },
        headers={**alpaca_headers, "Content-Type": "application/json"}
    )
    data = response.json()
    if "id" not in data:
        return f"Order failed: {data.get('message', 'Unknown error')}"
    return f"Paper trade placed: {side.upper()} {quantity} share(s) of {ticker} | Order ID: {data['id']}"

In [ ]:
print(place_order.invoke({"ticker": "AAPL", "side": "buy", "quantity": 1}))

Paper trade placed: BUY 1 share(s) of AAPL | Order ID: 7be11b3c-4918-4f3c-8661-f07b4090e08c


current_stock position  how many stocks we own right now

In [ ]:
response = requests.get(f"{ALPACA_BASE_URL}/v2/positions", headers=alpaca_headers)
print(response.json())

[{'asset_id': 'b0b6dd9d-8b9b-48a9-ba46-b9d54906e415', 'symbol': 'AAPL', 'exchange': 'NASDAQ', 'asset_class': 'us_equity', 'asset_marginable': True, 'qty': '2', 'qty_available': '2', 'avg_entry_price': '359', 'side': 'long', 'market_value': '716.46', 'cost_basis': '718', 'unrealized_pl': '-1.54', 'unrealized_plpc': '-0.00214', 'unrealized_intraday_pl': '-1.54', 'unrealized_intraday_plpc': '-0.00214', 'current_price': '358.23', 'lastday_price': '354.88', 'change_today': '0.00944'}]


In [ ]:
@tool
def get_stocks_owned() -> str:
    """Check what stocks we currently own and if we are making profit or loss."""
    response = requests.get(f"{ALPACA_BASE_URL}/v2/positions", headers=alpaca_headers)
    positions = response.json()
    if not positions:
        return "You do not own any stocks yet."
    result = "Stocks you own:\n"
    for p in positions:
        result += (
            f"  {p['symbol']}: {p['qty']} share(s) | "
            f"Bought at: ${float(p['avg_entry_price']):.2f} | "
            f"Profit/Loss: ${float(p['unrealized_pl']):.2f}\n"
        )
    return result

In [ ]:
print(get_stocks_owned.invoke({}))

Stocks you own:
  AAPL: 2 share(s) | Bought at: $359.00 | Profit/Loss: $-1.57



TO KNOW HOW MUCH BALANCE WE HAVE

In [ ]:
@tool
def get_balance() -> str:
    """Check how much virtual money we have left to buy stocks."""
    response = requests.get(f"{ALPACA_BASE_URL}/v2/account", headers=alpaca_headers)
    data = response.json()
    return (
        f"Total account value: ${float(data['equity']):,.2f} | "
        f"Available to spend: ${float(data['buying_power']):,.2f}"
    )

In [ ]:
print(get_balance.invoke({}))


print(get_stocks_owned.invoke({}))

Total account value: $99,998.43 | Available to spend: $398,417.50
Stocks you own:
  AAPL: 3 share(s) | Bought at: $358.75 | Profit/Loss: $-1.60



FINAL CODE

In [ ]:
system_prompt = """You are a professional stock trading analyst with the ability to place paper trades.

When asked to analyse a stock, follow these steps in order:
1. Get the current price → get_stock_price
2. Get the recent price history → get_price_history
3. Calculate RSI → calculate_rsi
4. Get the 50-day and 200-day SMA → get_sma
5. Get recent news headlines → get_company_news
6. Read the headlines and judge whether the news is Positive, Neutral, or Negative for the stock
7. Look at all the data together and output the analysis in this format:

═══════════════════════════════════════════
STOCK ANALYSIS REPORT: [TICKER]
═══════════════════════════════════════════

DECISION: BUY / SELL / HOLD
CONFIDENCE: High / Medium / Low

Current Price: $[price]
Recent trend: [describe direction from price history]

RSI(14): [value] → [Overbought / Neutral / Oversold]
SMA: [Golden Cross / Death Cross] — [short explanation]

News: [Positive / Neutral / Negative]
Key headlines:
  • [headline 1]
  • [headline 2]

Rationale: [2-3 sentences combining all signals to explain the decision]

Risk Note: This is a paper trading simulation for educational purposes only.
═══════════════════════════════════════════

EXECUTION RULE:
- DECISION is BUY with High or Medium confidence → call place_order to buy 1 share
- DECISION is SELL with High confidence and you already hold the stock → call place_order to sell 1 share
- DECISION is HOLD → do not place any order"""

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

google_api_key = userdata.get('API_KEY')
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

agent = create_agent(
    model=model,
    tools=[
        get_stock_price,
        get_price_history,
        calculate_rsi,
        get_sma,
        get_company_news,
        place_order,
        get_stocks_owned,
        get_balance
    ],
    system_prompt=system_prompt
)

In [ ]:
import pprint
response = agent.invoke({
    "messages": [{"role": "user", "content": "Should I buy Nvidia today?"}]
})
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I apologize, but I have run out of API credits to fetch the current stock price for Nvidia. Please try again in a minute.', 'extras': {'signature': 'CvwBAWkUfRPOmKkOhdzN0PeQj/0UwBlAg3U0zT1pEvwkx5HhaFZumap5gk2u+L4Txi5j6//GXQc5qW2oUfahMHIZGfVYW6gWIO+qQBbtBFv8ZDRW/CwNYgU1djp7gt7voY9uXpWApi14leW1w0vGAQIoqaIiWb3288UuG2lJHMf4wwZSRkIQNzQymbGQLdrmUi9xfvl3IMozV+n4gr/xO1Vq6bwip5qNJhjLMX/FNKmNl+SwyPFMWfkYob35ERShAGX/YhK/UiqLMFu/9G+JfpEKJC17Rj/qW/1ETpWG0BD/dEOxmiCWJ3BIJ0Av/NHjZqCQc3SZHdprPkoK4Tn2'}}]


In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get("API_KEY")

model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=api_key
)

response = model.invoke("What is observability in software?")
print(response.content)

Observability in software is the ability to understand the internal state of a system by examining its external outputs. In simpler terms, it's about being able to ask *any* question about what's happening inside your system without having to ship new code to answer that specific question.

It's a crucial concept for modern, complex, and distributed systems (like microservices) where it's impossible to predict every potential failure mode or performance bottleneck beforehand.

### Key Concepts:

1.  **Understanding "Why":** Observability goes beyond knowing *that* something is wrong (which is monitoring's job). It helps you understand *why* it's wrong, *what* caused it, and *how* to fix it.
2.  **Arbitrary Questions:** The hallmark of an observable system is its capacity to answer questions you didn't even know you'd need to ask when you built it. This is particularly valuable for "unknown unknowns."
3.  **Black Box vs. White Box:** An observable system is like a "white box" where you 

In [ ]:
def classify_news(headline: str) -> str:
    if "profit" in headline.lower() or "growth" in headline.lower():
        return "positive"
    elif "loss" in headline.lower() or "crash" in headline.lower():
        return "negative"
    return "neutral"

print(classify_news("Apple reports record App Store revenue"))

neutral


In [ ]:
from langsmith import traceable

@traceable
def classify_news(headline: str) -> str:
    if "profit" in headline.lower() or "growth" in headline.lower():
        return "positive"
    elif "loss" in headline.lower() or "crash" in headline.lower():
        return "negative"
    return "neutral"

print(classify_news("Apple reports record App Store revenue"))

neutral
